# 08 — Tokenizer Robustness: Is the TP Effect XLM-R-Specific?

Every TP number reported so far uses the XLM-RoBERTa tokenizer, because that is
the encoder COMET is built on. This notebook checks whether the TP inflation
under romanisation is a property of romanisation or a property of XLM-R, by
re-running the same measurement through three other tokenizers.

**Input:** `../data/indic/indic_parity_multi_tokenizer.xlsx`
**Output:** `../results/tables/tokenizer_robustness.csv`

> **Table 4.**
> They are computed here and registered with status `script` in
> `paper_numbers.yaml`, labelled as having no paper claim to check against.
> Treat them as a supporting analysis, not as a reproduction of a reported table.

## Step 0 — Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)


# ── Tokenizers present in the multi-tokenizer workbook ───────────────────────
TOKENIZERS = ["xlmr", "mbert", "gpt2", "byt5"]

## Step 1 — Confirm the Two Workbooks Agree

The multi-tokenizer workbook is a superset of the primary one: it adds
per-tokenizer columns but leaves the shared columns untouched. This is asserted
rather than assumed, because every earlier notebook reads the primary workbook
and this one reads the superset.

In [ ]:
SHARED = [COL_COMET_NAT, COL_COMET_ROM, COL_HUMAN,
          COL_TP_NAT, COL_TP_ROM, COL_IP_NAT, COL_IP_ROM]

print(f"{'Lang':>5}  {'rows (xlmr / multi)':>20}  {'shared columns identical':>26}")
print("-" * 56)
for lang in LANG_ORDER:
    a = pd.read_excel(DATA_XLMR, sheet_name=SHEET_MAP[lang])
    b = pd.read_excel(DATA_MULTI, sheet_name=SHEET_MAP[lang])
    same = all(
        np.allclose(pd.to_numeric(a[c], errors="coerce").fillna(-999),
                    pd.to_numeric(b[c], errors="coerce").fillna(-999), atol=1e-9)
        for c in SHARED
    )
    assert same, f"{lang}: shared columns diverge between the two workbooks"
    assert len(a) == len(b)
    print(f"{lang:>5}  {f'{len(a)} / {len(b)}':>20}  {str(same):>26}")

print("\n\u2713 The multi-tokenizer workbook is a strict superset of the primary one")

## Step 2 — Mean TP per Tokenizer, Native → Romanised

TP is defined the same way throughout: target token count divided by English
source token count, under the tokenizer named in the column.

In [ ]:
rows = []
for tok in TOKENIZERS:
    print(f"\n  {tok}")
    print(f"  {'Lang':>5}  {'TP_nat':>9}  {'TP_rom':>9}  {'\u0394TP%':>8}")
    print("  " + "-" * 36)
    for lang in LANG_ORDER:
        b = pd.read_excel(DATA_MULTI, sheet_name=SHEET_MAP[lang])
        n = pd.to_numeric(b[f"Translation_{tok}_TP"], errors="coerce").mean()
        r = pd.to_numeric(
            b[f"Translation_Transliteration_romanized_{tok}_TP"], errors="coerce").mean()
        pct = (r / n - 1) * 100
        rows.append(dict(tokenizer=tok, lang=lang, tp_nat=n, tp_rom=r, tp_pct=pct))
        print(f"  {lang:>5}  {n:>9.3f}  {r:>9.3f}  {pct:>+8.1f}")

tok_table = pd.DataFrame(rows)

## Step 3 — The Direction of the Effect Is Not Universal

XLM-R is the outlier, and it is the one that matters for COMET.

In [ ]:
print(f"{'Tokenizer':>10}  {'\u0394TP% range':>22}  {'direction':>28}")
print("-" * 64)
for tok in TOKENIZERS:
    s = tok_table[tok_table.tokenizer == tok]["tp_pct"]
    if (s > 0).all():
        direction = "inflates for all 5"
    elif (s < 0).all():
        direction = "collapses for all 5"
    else:
        direction = "mixed"
    print(f"{tok:>10}  {f'{s.min():+.1f}% to {s.max():+.1f}%':>22}  {direction:>28}")

xlmr = tok_table[tok_table.tokenizer == "xlmr"]["tp_pct"]
assert (xlmr > 0).all(), "XLM-R TP should inflate under romanisation for all five"
print("\n\u2713 XLM-R (the COMET encoder) inflates TP for all five languages")
print("  Byte- and BPE-level tokenizers (gpt2, byt5) move the other way, because")
print("  they already fragment Indic scripts into bytes or characters; romanisation")
print("  shortens their sequences. mBERT is mixed. The TP inflation reported in")
print("  notebook 02 is therefore a property of XLM-R's vocabulary allocation —")
print("  which is exactly the encoder whose scores the paper is auditing.")

## Step 4 — Save

In [ ]:
path = TABLES_DIR / "tokenizer_robustness.csv"
tok_table.to_csv(path, index=False)
print(tok_table.round(3).to_string(index=False))
print(f"\nSaved \u2192 {path}")

## Step 5 — Output Manifest

In [ ]:
print("=== Notebook 08 — output manifest ===")
print("  tokenizer_robustness.csv")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1